# Notebook 08 — Valutazione di Sistemi RAG

**Settimana:** 8
**Blocco:** 4.5 — Valutazione e miglioramento della pipeline
**Tempo stimato:** 3,5-4 ore

## Cosa costruirai

- Un valutatore di faithfulness basato su LLM (giudice LLM)
- Un generatore automatico di testset a partire dai documenti
- Una pipeline di valutazione completa che assegna un punteggio al sistema RAG della settimana 7
- Un esperimento A/B: confronta la pipeline base con la pipeline con query rewriting
- **Sfida finale:** cruscotto di valutazione che rileva le domande problematiche

## Prerequisiti

Settimana 7 completata. La collezione `rag_docs` deve esistere in `./chroma_db`.

```bash
pip install ragas langchain-openai datasets chromadb openai python-dotenv
```

---
## Cella 0 — Setup

In [ ]:
import os
import json
import time
import uuid
import datetime
import statistics
import numpy as np
from dataclasses import dataclass, field, asdict
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()

cliente_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma_cliente = chromadb.PersistentClient(path="./chroma_db")

MODELO_EMBEDDING  = "text-embedding-3-small"
MODELO_GENERACION = "gpt-4o-mini"
MODELO_EVALUADOR  = "gpt-4o-mini"   # stesso modello — in produzione preferiresti gpt-4o

# ── Helper dalle settimane precedenti ────────────────────────────────────────

def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model=MODELO_EMBEDDING, input=texto)
    return r.data[0].embedding

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1024
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_GENERACION, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.0
) -> dict:
    """Come completar() ma forza la JSON mode e fa il parsing del risultato."""
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente_openai.chat.completions.create(
        model=MODELO_EVALUADOR, messages=msgs,
        temperature=temperature,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

# ── Carica la collezione del notebook 07 ─────────────────────────────────────
try:
    coleccion_rag = chroma_cliente.get_collection("rag_docs")
    print(f"Collezione 'rag_docs' caricata: {coleccion_rag.count()} chunk")
except Exception as e:
    print(f"ERRORE: la collezione 'rag_docs' non esiste. Esegui prima il notebook 07.")
    print(f"  Dettaglio: {e}")

Path("evaluaciones").mkdir(exist_ok=True)
print("Setup completato.")

---
## Cella 1 — Pipeline RAG di base (dal notebook 07)

Riproduci qui la pipeline base e quella avanzata per valutarle entrambe.
Aggiungi un logging completo a ogni query.

In [ ]:
UMBRAL_RELEVANCIA = 0.65

SYSTEM_RAG = """You are a technical documentation assistant.
Answer questions using ONLY the information provided in the context.
If the answer is not found in the context, say exactly: 'INSUFFICIENT_CONTEXT'
Do not use external knowledge. Be concise and factual."""


@dataclass
class TrazaRAG:
    """Registro completo di una query RAG, per la diagnostica e la valutazione."""
    id:                str
    timestamp:         str
    pregunta:          str
    query_retrieval:   str            # può differire da pregunta se c'è rewriting
    chunks_recuperados: list[str]
    chunks_scores:     list[float]
    chunks_fuentes:    list[str]
    respuesta:         str
    n_chunks_usados:   int
    respondida:        bool
    latencia_ms:       float
    modelo_embedding:  str
    modelo_generacion: str
    # Metriche di valutazione (vengono compilate dopo)
    faithfulness:      float | None = None
    context_relevance: float | None = None
    answer_relevance:  float | None = None


def recuperar(
    query: str,
    coleccion: chromadb.Collection,
    k: int = 4
) -> list[dict]:
    vector = generar_embedding(query)
    resp   = coleccion.query(
        query_embeddings=[vector],
        n_results=min(k, coleccion.count()),
        include=["documents", "distances", "metadatas"]
    )
    return [
        {"texto": doc, "similitud": round(1 - dist, 4), "meta": meta}
        for doc, dist, meta in zip(
            resp["documents"][0], resp["distances"][0], resp["metadatas"][0]
        )
    ]


def construir_prompt(pregunta: str, chunks: list[dict]) -> str:
    bloques = [
        f"[Context {i} | Source: {c['meta'].get('fuente','?')} | Score: {c['similitud']:.2f}]\n{c['texto']}"
        for i, c in enumerate(chunks, 1)
    ]
    return f"CONTEXT:\n{'\n\n---\n\n'.join(bloques)}\n\nQUESTION: {pregunta}\n\nANSWER:"


def reescribir_query(pregunta: str) -> str:
    prompt = (f"Rewrite as an optimized technical search query (only the query):\n"
              f"Question: {pregunta}\nOptimized query:")
    return completar([{"role": "user", "content": prompt}], temperature=0.0).strip()


def responder_con_traza(
    pregunta: str,
    coleccion: chromadb.Collection,
    k: int = 4,
    umbral: float = UMBRAL_RELEVANCIA,
    usar_rewriting: bool = False
) -> TrazaRAG:
    """Pipeline RAG che restituisce la risposta + la tracciabilità completa."""
    t_inicio           = time.perf_counter()
    query_retrieval    = reescribir_query(pregunta) if usar_rewriting else pregunta

    chunks_todos       = recuperar(query_retrieval, coleccion, k=k)
    chunks_relevantes  = [c for c in chunks_todos if c["similitud"] >= umbral]

    if not chunks_relevantes:
        respuesta    = "INSUFFICIENT_CONTEXT"
        respondida   = False
    else:
        prompt       = construir_prompt(pregunta, chunks_relevantes)
        respuesta    = completar(
            [{"role": "user", "content": prompt}], system=SYSTEM_RAG
        )
        respondida   = "INSUFFICIENT_CONTEXT" not in respuesta

    latencia_ms = (time.perf_counter() - t_inicio) * 1000

    return TrazaRAG(
        id=str(uuid.uuid4())[:8],
        timestamp=datetime.datetime.now().isoformat(),
        pregunta=pregunta,
        query_retrieval=query_retrieval,
        chunks_recuperados=[c["texto"] for c in chunks_relevantes],
        chunks_scores=[c["similitud"] for c in chunks_relevantes],
        chunks_fuentes=[c["meta"].get("fuente", "?") for c in chunks_relevantes],
        respuesta=respuesta,
        n_chunks_usados=len(chunks_relevantes),
        respondida=respondida,
        latencia_ms=round(latencia_ms, 1),
        modelo_embedding=MODELO_EMBEDDING,
        modelo_generacion=MODELO_GENERACION,
    )


# ── Test rapido ────────────────────────────────────────────────────────────────
traza_prueba = responder_con_traza(
    "What are the main features of FastAPI?", coleccion_rag
)
print(f"Domanda:    {traza_prueba.pregunta}")
print(f"Risposta data: {traza_prueba.respondida}")
print(f"Chunk:      {traza_prueba.n_chunks_usados}, score: {traza_prueba.chunks_scores}")
print(f"Latenza:    {traza_prueba.latencia_ms} ms")
print(f"Risposta:   {traza_prueba.respuesta[:200]}...")

---
## Cella 2 — Generatore di testset sintetico

Genera domande di valutazione a partire dai chunk stessi indicizzati.
Ogni domanda arriva con la sua risposta attesa (ground truth) derivata dal chunk.
Il risultato viene salvato in JSON per riutilizzarlo in tutti gli esperimenti.

In [ ]:
@dataclass
class EjemploTest:
    id:               str
    pregunta:         str
    respuesta_gold:   str    # risposta corretta nota
    contexto_gold:    str    # chunk da cui è stata derivata la domanda
    fuente:           str    # articolo di Wikipedia
    tipo:             str    # factual | conceptual | comparativa


SYSTEM_GENERADOR = """You are creating evaluation questions for a RAG system.
Generate questions that can ONLY be answered using the provided text.
Avoid generic questions that any LLM could answer from training data.
Focus on specific facts, numbers, definitions, or relationships in the text."""


def generar_ejemplos_de_chunk(
    chunk: str,
    fuente: str,
    n_preguntas: int = 2
) -> list[EjemploTest]:
    """Genera n_preguntas coppie (domanda, risposta_gold) a partire da un chunk."""
    prompt = f"""Generate {n_preguntas} evaluation question-answer pairs from this text.
Each pair should test a specific fact or concept in the text.
Mix question types: factual (what/who/when), conceptual (how/why), and comparative (difference between).

TEXT:
{chunk}

Return a JSON object with key \"pairs\" containing a list where each item has:
- \"question\": the question
- \"answer\": the expected answer (concise, 1-3 sentences)
- \"type\": \"factual\" | \"conceptual\" | \"comparative\""""

    try:
        datos = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_GENERADOR)
        ejemplos = []
        for par in datos.get("pairs", []):
            ejemplos.append(EjemploTest(
                id=str(uuid.uuid4())[:8],
                pregunta=par["question"],
                respuesta_gold=par["answer"],
                contexto_gold=chunk,
                fuente=fuente,
                tipo=par.get("type", "factual")
            ))
        return ejemplos
    except Exception as e:
        print(f"  Errore nella generazione delle domande: {e}")
        return []


def construir_testset(
    coleccion: chromadb.Collection,
    n_chunks_muestrear: int = 15,
    preguntas_por_chunk: int = 2,
    ruta_cache: str = "evaluaciones/testset.json"
) -> list[EjemploTest]:
    """
    Costruisce un testset campionando chunk dalla collezione.
    Mette in cache in JSON per non rigenerarlo a ogni esperimento.
    """
    if Path(ruta_cache).exists():
        print(f"Caricamento del testset dalla cache: {ruta_cache}")
        with open(ruta_cache) as f:
            datos = json.load(f)
        return [EjemploTest(**d) for d in datos]

    print(f"Generazione del testset ({n_chunks_muestrear} chunk × {preguntas_por_chunk} domande)...")

    # Ottiene tutti i chunk della collezione
    todos = coleccion.get(include=["documents", "metadatas"])
    docs  = todos["documents"]
    metas = todos["metadatas"]

    # Filtra i chunk con contenuto sufficiente (scarta quelli troppo corti)
    pares = [(d, m) for d, m in zip(docs, metas) if len(d) > 200]

    # Campiona senza ripetizione
    np.random.seed(42)
    indices   = np.random.choice(len(pares), size=min(n_chunks_muestrear, len(pares)), replace=False)
    muestra   = [pares[i] for i in indices]

    testset = []
    for i, (chunk, meta) in enumerate(muestra):
        fuente = meta.get("fuente", "desconocida")
        print(f"  [{i+1}/{len(muestra)}] {fuente[:30]}...")
        ejemplos = generar_ejemplos_de_chunk(chunk, fuente, preguntas_por_chunk)
        testset.extend(ejemplos)

    # Salva in cache
    with open(ruta_cache, "w") as f:
        json.dump([asdict(e) for e in testset], f, indent=2)

    print(f"\nTestset generato: {len(testset)} esempi → salvato in '{ruta_cache}'")
    return testset


testset = construir_testset(coleccion_rag, n_chunks_muestrear=15, preguntas_por_chunk=2)

# Distribuzione del testset
from collections import Counter
tipos = Counter(e.tipo for e in testset)
fuentes = Counter(e.fuente for e in testset)
print(f"\nDistribuzione per tipo:   {dict(tipos)}")
print(f"Distribuzione per fonte:  {dict(fuentes)}")

# Mostra 3 esempi
print("\nEsempi:")
for e in testset[:3]:
    print(f"  [{e.tipo}] {e.pregunta}")
    print(f"    → Gold: {e.respuesta_gold[:80]}...")

---
## Cella 3 — Valutatore LLM: faithfulness e context relevance

Costruisci il giudice LLM che valuta ogni risposta generata.
Faithfulness misura se la risposta inventa cose; context relevance misura se i chunk recuperati sono utili.

In [ ]:
@dataclass
class ResultadoEvaluacion:
    traza_id:          str
    pregunta:          str
    respuesta:         str
    respondida:        bool
    faithfulness:      float | None    # 0-1, None se non risposta
    context_relevance: float           # media degli score di similarità
    answer_relevance:  float | None    # 0-1, stimato dall'LLM
    latencia_ms:       float
    notas_faithfulness: list[str] = field(default_factory=list)


SYSTEM_FAITHFULNESS = """You are a strict fact-checker evaluating whether a generated answer
is supported by the provided context. Be precise and critical."""

PROMPT_FAITHFULNESS = """Evaluate whether the answer is faithful to the context.

CONTEXT:
{contexto}

QUESTION: {pregunta}
ANSWER: {respuesta}

For each concrete factual claim in the answer, check if it is supported by the context.
Return a JSON object with:
- "claims": list of objects with {"claim": "...", "supported": true/false, "evidence": "quote or null"}
- "faithfulness_score": float 0-1 (supported_claims / total_claims)
- "verdict": "FAITHFUL" | "PARTIALLY_FAITHFUL" | "UNFAITHFUL"
- "main_issue": one sentence about the biggest faithfulness problem, or null if fully faithful"""


def evaluar_faithfulness(traza: TrazaRAG) -> dict:
    """Valuta se la risposta si basa solo sul contesto. Restituisce uno score 0-1."""
    if not traza.respondida:
        return {"faithfulness_score": None, "verdict": "NOT_ANSWERED", "claims": [], "main_issue": None}

    contexto = "\n\n---\n\n".join(traza.chunks_recuperados)
    prompt   = PROMPT_FAITHFULNESS.format(
        contexto=contexto[:3000],   # limita per non eccedere il contesto
        pregunta=traza.pregunta,
        respuesta=traza.respuesta
    )
    return completar_json(
        [{"role": "user", "content": prompt}],
        system=SYSTEM_FAITHFULNESS
    )


PROMPT_ANSWER_RELEVANCE = """Evaluate how relevant this answer is to the question.

QUESTION: {pregunta}
ANSWER: {respuesta}

Return JSON with:
- "relevance_score": float 0-1 (0=completely irrelevant, 1=perfectly relevant)
- "reason": one sentence explanation"""


def evaluar_answer_relevance(traza: TrazaRAG) -> dict:
    """Valuta se la risposta risponde alla domanda (ignora se è corretta o meno)."""
    if not traza.respondida:
        return {"relevance_score": None, "reason": "No answer provided"}

    prompt = PROMPT_ANSWER_RELEVANCE.format(
        pregunta=traza.pregunta,
        respuesta=traza.respuesta[:500]
    )
    return completar_json([{"role": "user", "content": prompt}])


def evaluar_traza(traza: TrazaRAG) -> ResultadoEvaluacion:
    """Valuta una traccia completa: faithfulness + context relevance + answer relevance."""

    # Context relevance: media degli score di similarità (non richiede l'LLM)
    context_relevance = (
        statistics.mean(traza.chunks_scores) if traza.chunks_scores else 0.0
    )

    # Faithfulness: chiama l'LLM valutatore
    eval_faith = evaluar_faithfulness(traza)

    # Answer relevance: chiama l'LLM valutatore
    eval_rel   = evaluar_answer_relevance(traza)

    return ResultadoEvaluacion(
        traza_id=traza.id,
        pregunta=traza.pregunta,
        respuesta=traza.respuesta,
        respondida=traza.respondida,
        faithfulness=eval_faith.get("faithfulness_score"),
        context_relevance=round(context_relevance, 4),
        answer_relevance=eval_rel.get("relevance_score"),
        latencia_ms=traza.latencia_ms,
        notas_faithfulness=[
            c["claim"] for c in eval_faith.get("claims", []) if not c.get("supported", True)
        ]
    )


# ── Test del valutatore su 3 esempi ───────────────────────────────────────────
print("Valutazione di 3 esempi del testset...")
print("(Questo fa ~6 chiamate all'LLM valutatore)\n")

evaluaciones_prueba = []
for ejemplo in testset[:3]:
    traza = responder_con_traza(ejemplo.pregunta, coleccion_rag)
    eval_r = evaluar_traza(traza)
    evaluaciones_prueba.append(eval_r)

    print(f"Domanda:           {ejemplo.pregunta[:60]}...")
    print(f"Risposta data:     {eval_r.respondida}")
    print(f"Faithfulness:      {eval_r.faithfulness}")
    print(f"Context relevance: {eval_r.context_relevance}")
    print(f"Answer relevance:  {eval_r.answer_relevance}")
    if eval_r.notas_faithfulness:
        print(f"Affermazioni non supportate: {eval_r.notas_faithfulness}")
    print()

---
## Cella 4 — Valutazione completa ed esperimento A/B

Esegui il testset completo con la pipeline base e con il query rewriting.
Confronta le metriche per decidere se il rewriting vale la pena.

In [ ]:
def evaluar_pipeline(
    testset: list[EjemploTest],
    coleccion: chromadb.Collection,
    usar_rewriting: bool = False,
    nombre: str = "pipeline",
    ruta_cache: str | None = None
) -> list[ResultadoEvaluacion]:
    """Valuta la pipeline sul testset completo. Mette in cache i risultati."""

    if ruta_cache and Path(ruta_cache).exists():
        print(f"[{nombre}] Caricamento della valutazione dalla cache: {ruta_cache}")
        with open(ruta_cache) as f:
            datos = json.load(f)
        return [ResultadoEvaluacion(**d) for d in datos]

    print(f"[{nombre}] Valutazione di {len(testset)} domande (rewriting={usar_rewriting})...")
    resultados = []

    for i, ejemplo in enumerate(testset):
        print(f"  [{i+1}/{len(testset)}] {ejemplo.pregunta[:50]}...", end=" ")

        traza  = responder_con_traza(
            ejemplo.pregunta, coleccion,
            usar_rewriting=usar_rewriting
        )
        eval_r = evaluar_traza(traza)
        resultados.append(eval_r)

        estado = f"F={eval_r.faithfulness:.2f}" if eval_r.faithfulness is not None else "no-resp"
        print(f"→ {estado}")

    if ruta_cache:
        with open(ruta_cache, "w") as f:
            json.dump([asdict(r) for r in resultados], f, indent=2)
        print(f"  Salvato in '{ruta_cache}'")

    return resultados


def resumen_metricas(resultados: list[ResultadoEvaluacion], nombre: str = "") -> dict:
    """Calcola le statistiche aggregate dell'insieme di valutazioni."""
    respondidas = [r for r in resultados if r.respondida]
    n_total     = len(resultados)
    n_resp      = len(respondidas)

    def media(valores):
        limpios = [v for v in valores if v is not None]
        return round(statistics.mean(limpios), 4) if limpios else None

    return {
        "nombre":              nombre,
        "n_total":             n_total,
        "n_respondidas":       n_resp,
        "tasa_respuesta":      round(n_resp / n_total, 3),
        "faithfulness_media":  media([r.faithfulness for r in respondidas]),
        "context_rel_media":   media([r.context_relevance for r in respondidas]),
        "answer_rel_media":    media([r.answer_relevance for r in respondidas]),
        "latencia_media_ms":   round(statistics.mean([r.latencia_ms for r in resultados]), 1),
        "n_con_alucinaciones": sum(
            1 for r in respondidas
            if r.faithfulness is not None and r.faithfulness < 0.7
        )
    }


# ── Esperimento A/B ─────────────────────────────────────────────────────────────
# ATTENZIONE: questo fa ~4 chiamate all'LLM per domanda (1 retrieval + 1 generazione + 2 valutazione)
# Con un testset di 30 domande × 2 configurazioni = ~240 chiamate. Costo ~$0,05 con gpt-4o-mini.

# Usa solo le prime 10 domande per il test (decommenta tutto il testset quando vuoi)
TESTSET_REDUCIDO = testset[:10]

resultados_basico  = evaluar_pipeline(
    TESTSET_REDUCIDO, coleccion_rag,
    usar_rewriting=False,
    nombre="Base",
    ruta_cache="evaluaciones/resultados_basico.json"
)

resultados_rewrite = evaluar_pipeline(
    TESTSET_REDUCIDO, coleccion_rag,
    usar_rewriting=True,
    nombre="Con rewriting",
    ruta_cache="evaluaciones/resultados_rewrite.json"
)

# ── Tabella comparativa ────────────────────────────────────────────────────────
res_basico  = resumen_metricas(resultados_basico,  "Base")
res_rewrite = resumen_metricas(resultados_rewrite, "Con rewriting")

print("\n" + "=" * 62)
print(f"{'METRICA':<30} {'BASE':>12} {'CON REWRITING':>14}")
print("=" * 62)

metricas = [
    ("Tasso di risposta",       "tasa_respuesta"),
    ("Faithfulness media",      "faithfulness_media"),
    ("Context relevance media", "context_rel_media"),
    ("Answer relevance media",  "answer_rel_media"),
    ("Latenza media (ms)",      "latencia_media_ms"),
    ("Risposte con allucinaz.", "n_con_alucinaciones"),
]

for nombre, clave in metricas:
    v_b = res_basico.get(clave, "—")
    v_r = res_rewrite.get(clave, "—")
    # Indica quale è migliore (↑ per le metriche di qualità, ↓ per latenza e allucinazioni)
    if isinstance(v_b, float) and isinstance(v_r, float):
        if clave in ("latencia_media_ms",):
            indicador = "↑" if v_r < v_b else ("↓" if v_r > v_b else "=")
        elif clave == "n_con_alucinaciones":
            indicador = "✓" if v_r < v_b else ("✗" if v_r > v_b else "=")
        else:
            indicador = "↑" if v_r > v_b else ("↓" if v_r < v_b else "=")
        print(f"{nombre:<30} {str(v_b):>12} {str(v_r):>12} {indicador}")
    else:
        print(f"{nombre:<30} {str(v_b):>12} {str(v_r):>14}")

print("=" * 62)

**Analisi dell'esperimento A/B** (modifica questa cella):

- Il query rewriting ha migliorato la faithfulness?
- La latenza extra del rewriting si giustifica con il miglioramento della qualità?
- Quale metrica è cambiata di più? Perché?
- Cosa cambieresti in seguito per migliorare il sistema?

---
## Sfida finale — Cruscotto di valutazione

Costruisci un rilevatore di domande problematiche che classifica ogni query
del testset in base al suo tipo di fallimento, per dare priorità ai miglioramenti.

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
from enum import Enum

class TipoFallo(str, Enum):
    SIN_FALLO          = "sin_fallo"           # faithfulness >= 0.9 e risposta data
    ALUCINACION        = "alucinacion"          # faithfulness < 0.7 (risponde ma inventa)
    FIDELIDAD_PARCIAL  = "fidelidad_parcial"    # 0.7 <= faithfulness < 0.9
    SIN_RESPUESTA      = "sin_respuesta"        # non è arrivata a rispondere (nessun chunk rilevante)
    BAJA_RELEVANCIA    = "baja_relevancia"      # risposta data ma context_relevance < 0.70


def clasificar_fallo(resultado: ResultadoEvaluacion) -> TipoFallo:
    """
    Classifica il tipo di fallimento di una valutazione.

    Priorità di classificazione:
    1. Se non risposta → SIN_RESPUESTA
    2. Se context_relevance < 0.70 → BAJA_RELEVANCIA
    3. Se faithfulness < 0.70 → ALUCINACION
    4. Se 0.70 <= faithfulness < 0.90 → FIDELIDAD_PARCIAL
    5. Se faithfulness >= 0.90 → SIN_FALLO
    """
    # TODO: implementa la logica di classificazione
    raise NotImplementedError("Implementa clasificar_fallo()")


def cuadro_de_mando(
    resultados: list[ResultadoEvaluacion],
    nombre_experimento: str = "pipeline"
) -> None:
    """
    Mostra:
    1. Distribuzione dei tipi di fallimento
    2. I 3 casi peggiori (più urgenti da correggere)
    3. Raccomandazioni di miglioramento per tipo di fallimento
    """
    # TODO: implementa il cruscotto
    #
    # Passo 1: classifica tutti i risultati
    # clasificados = [(r, clasificar_fallo(r)) for r in resultados]
    #
    # Passo 2: conta per tipo di fallimento
    # conteo = Counter(tipo for _, tipo in clasificados)
    #
    # Passo 3: mostra la distribuzione con le percentuali
    #
    # Passo 4: mostra i 3 casi con la faithfulness più bassa
    # (solo quelli con risposta data)
    #
    # Passo 5: mostra le raccomandazioni per tipo di fallimento
    # RECOMENDACIONES = {
    #     TipoFallo.ALUCINACION:        "Rafforza l'istruzione di grounding nel system prompt",
    #     TipoFallo.FIDELIDAD_PARCIAL:  "Verifica se il prompt permette di rispondere con conoscenza esterna",
    #     TipoFallo.SIN_RESPUESTA:      "Abbassa la soglia di rilevanza o aumenta k",
    #     TipoFallo.BAJA_RELEVANCIA:    "Rivedi la strategia di chunking o il modello di embedding",
    #     TipoFallo.SIN_FALLO:          "Nessuna azione necessaria",
    # }

    raise NotImplementedError("Implementa cuadro_de_mando()")


# Decommenta quando hai implementato le funzioni
# print("\n" + "=" * 60)
# print("CRUSCOTTO — Pipeline Base")
# cuadro_de_mando(resultados_basico, "Base")

# print("\n" + "=" * 60)
# print("CRUSCOTTO — Con Query Rewriting")
# cuadro_de_mando(resultados_rewrite, "Con rewriting")

### Estensioni opzionali

1. **RAGAS ufficiale:** usa la libreria `ragas` per replicare l'esperimento e confronta i suoi score con quelli del tuo valutatore LLM.
   ```python
   from ragas import evaluate
   from ragas.metrics import faithfulness, answer_relevancy
   from datasets import Dataset
   ```

2. **Soglia ottimale:** sperimenta con diversi valori di `UMBRAL_RELEVANCIA` (0.55, 0.60, 0.65, 0.70, 0.75) e traccia un grafico del tasso di risposta rispetto alla faithfulness media per trovare la soglia ottimale.

3. **Valutazione automatica in CI:** scrivi uno script Python che carichi il testset, esegua la pipeline e fallisca se la faithfulness media scende sotto 0.80. Usalo come gate di qualità prima di distribuire modifiche.

In [ ]:
# Estensione 2: ricerca della soglia ottimale
# Decommenta ed esegui quando hai completato la sfida principale

# UMBRALES = [0.55, 0.60, 0.65, 0.70, 0.75]
# resultados_umbral = []

# for umbral in UMBRALES:
#     trazas = [responder_con_traza(e.pregunta, coleccion_rag, umbral=umbral) for e in TESTSET_REDUCIDO]
#     n_resp = sum(1 for t in trazas if t.respondida)
#     resultados_umbral.append({
#         "umbral": umbral,
#         "tasa_respuesta": n_resp / len(trazas),
#     })
#     print(f"Soglia {umbral}: {n_resp}/{len(trazas)} con risposta ({n_resp/len(trazas):.0%})")

# Per il grafico:
# import matplotlib.pyplot as plt
# plt.plot([r["umbral"] for r in resultados_umbral],
#          [r["tasa_respuesta"] for r in resultados_umbral], marker="o")
# plt.xlabel("Soglia di rilevanza")
# plt.ylabel("Tasso di risposta")
# plt.title("Tradeoff: soglia di rilevanza vs copertura")
# plt.savefig("evaluaciones/umbral_vs_cobertura.png")
# plt.show()

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
## Chiusura del blocco RAG

Hai completato le quattro settimane del blocco RAG:

- **S05** Embedding e ricerca semantica
- **S06** Database vettoriali e chunking
- **S07** Pipeline RAG completa con query rewriting
- **S08** Valutazione e miglioramento iterativo ← qui

La settimana 9 inizia il **Blocco 5: Agenti e Tool Use avanzato**.
Costruirai il primo agente che decide quali strumenti usare e quando fermarsi.

---
*Notebook 08 — Settimana 8 del Piano di Formazione AI Engineering*